# Train, evaluate, and extrapolate a SpecMine baseline

This notebook trains on nonblank supplied labels, weighting trainable models by confidence / 100, and evaluates on repository-disjoint data. It then refits the selected model on the available labeled specs and predicts an unlabeled sample. Metadata and features are cache-first; the two Markdown models also reuse cached Markdown and download missing files unless `LOCAL_DATA_ONLY` is enabled. Unavailable labeled Markdown is excluded and reported. All IDs in the labels CSV, including blank labels, are excluded from prediction. Predictions are not verified authorship labels.


In [ ]:
import logging
import os
import sys
import tempfile
import uuid
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone

from specmine.data import (
    MARKDOWN_COLUMN,
    load_data,
    load_feature_data,
    load_spec_markdown,
)
from specmine.evaluate import escape_spreadsheet_formula, predict_model
from specmine.features import build_features
from specmine.labels import SPEC_ID_COLUMN, read_labels_csv
from specmine.train import (
    AVAILABLE_MODELS,
    DEFAULT_MODELS,
    DEFAULT_TEXT_FEATURE,
    ID_COLUMN,
    REPOSITORY_COLUMN,
    TEXT_MODELS,
    fit_baseline_model,
    prepare_training_data,
    select_model_inputs,
    train_models,
)

project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "src" / "specmine").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not find src/specmine. Open this notebook from the project "
        "folder or one of its subfolders."
    )
sys.path.insert(0, str(project_root / "src"))

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logging.getLogger("specmine.train").setLevel(logging.INFO)

SEED = 42
TEST_SIZE = 0.2
MODEL_NAME = "logistic_regression"  # See AVAILABLE_MODELS for all choices.
TEXT_FEATURE = DEFAULT_TEXT_FEATURE  # Used only by markdown_plus_feature.
PREDICTION_SAMPLE_SIZE = 100
LOCAL_DATA_ONLY = False
DATA_DIR = project_root / "notebooks" / "data"
LABELS_CSV = project_root / "examples" / "real_labels.csv"
MODEL_DIR = project_root / "models"
PREDICTIONS_CSV = project_root / "notebooks" / "unlabeled_predictions.csv"

print(f"Labels CSV: {LABELS_CSV}")
print(f"Prediction output base: {PREDICTIONS_CSV}")

## Load labeled data

`LABELS_CSV` points to `examples/real_labels.csv`; its `reasoning` column is dropped on load. Labels are AI-assigned, so scores measure agreement with them. Feature rows are loaded only for nonblank labeled specs; selecting a text model also loads their Markdown and reports unavailable files. Training needs labeled examples from at least two repositories.


In [ ]:
if MODEL_NAME not in AVAILABLE_MODELS:
    raise ValueError(f"MODEL_NAME must be one of {AVAILABLE_MODELS}.")
if PREDICTION_SAMPLE_SIZE < 1:
    raise ValueError("PREDICTION_SAMPLE_SIZE must be a positive integer.")

labels_frame = read_labels_csv(LABELS_CSV)
metadata = load_data(seed=SEED, data_dir=DATA_DIR, local_data_only=LOCAL_DATA_ONLY)
if metadata is None:
    raise RuntimeError("SpecMine metadata loading failed; training cannot continue.")

training_features, training_labels, training_repositories, training_confidences = (
    prepare_training_data(
        metadata,
        labels_frame,
        seed=SEED,
        include_markdown=MODEL_NAME in TEXT_MODELS,
        data_dir=DATA_DIR,
        local_data_only=LOCAL_DATA_ONLY,
    )
)
if training_repositories.nunique() < 2:
    raise ValueError(
        "Training needs labeled specs from at least two repositories. "
        "Review another repository in notebooks/data_init.ipynb, then "
        "add its labels to the CSV."
    )
print(f"Labeled specs used for training/evaluation: {len(training_features)}")
print(f"Labeled repositories: {training_repositories.nunique()}")
print(f"Label counts: {training_labels.value_counts().sort_index().to_dict()}")

## Evaluate on a repository-disjoint holdout

The numeric baselines and the selected model are trained on the same available labeled rows and saved under `models/`. `MODEL_NAME` selects the model subsequently refit on all available labeled specs; `TEXT_FEATURE` chooses the numeric input for `markdown_plus_feature`. Holdout metrics describe agreement with supplied labels, not verified authorship accuracy.


In [ ]:
selected_models = list(DEFAULT_MODELS)
if MODEL_NAME not in selected_models:
    selected_models.append(MODEL_NAME)
training_results = train_models(
    training_features,
    training_labels,
    training_repositories,
    confidence_scores=training_confidences,
    model_names=selected_models,
    text_feature=TEXT_FEATURE,
    test_size=TEST_SIZE,
    seed=SEED,
    model_dir=MODEL_DIR,
)
holdout_result = training_results[MODEL_NAME]
print(f"Saved holdout baselines to: {MODEL_DIR}")
print(
    f"{MODEL_NAME}: {holdout_result['n_train']} train specs from "
    f"{holdout_result['n_train_repositories']} repositories; "
    f"{holdout_result['n_test']} holdout specs from "
    f"{holdout_result['n_test_repositories']} repositories"
)
display(pd.DataFrame([holdout_result["metrics"]], index=[MODEL_NAME]))

## Refit on labeled data and predict unlabeled specs

The selected estimator is cloned and fit on all available nonblank labeled examples. Unlabeled candidates without required features or Markdown are replaced; the notebook stops without writing predictions if it cannot fill the requested sample. The uniform baseline returns probabilities of 0.5 and seeded random class guesses.


In [ ]:
reviewed_ids = set(labels_frame[SPEC_ID_COLUMN].astype("string"))
metadata_ids = metadata[ID_COLUMN].astype("string")
prediction_pool = metadata.loc[~metadata_ids.isin(reviewed_ids)].dropna(
    subset=[ID_COLUMN, REPOSITORY_COLUMN]
)
prediction_pool = prediction_pool.drop_duplicates(subset=[ID_COLUMN])
if len(prediction_pool) < PREDICTION_SAMPLE_SIZE:
    raise ValueError(
        f"Only {len(prediction_pool)} unlabeled specs are available; "
        f"{PREDICTION_SAMPLE_SIZE} are required."
    )

feature_columns = holdout_result["feature_columns"]
requires_markdown = MARKDOWN_COLUMN in feature_columns
requires_numeric = any(column != MARKDOWN_COLUMN for column in feature_columns)
candidate_order = prediction_pool.sample(frac=1, random_state=SEED).reset_index(
    drop=True
)
target_count = PREDICTION_SAMPLE_SIZE
selected_metadata_parts = []
selected_feature_parts = []
selected_markdown_parts = []
unavailable_features = 0
unavailable_markdown = 0
cursor = 0
selected_count = 0

while selected_count < target_count and cursor < len(candidate_order):
    needed = target_count - selected_count
    requested_batch = needed if requires_markdown else max(needed, 25)
    batch_size = min(requested_batch, len(candidate_order) - cursor)
    candidate_batch = candidate_order.iloc[cursor : cursor + batch_size]
    cursor += batch_size
    matched_candidates = candidate_batch

    if requires_numeric:
        feature_batch = load_feature_data(
            candidate_batch[[ID_COLUMN]],
            data_dir=DATA_DIR,
            local_data_only=LOCAL_DATA_ONLY,
        )
        if feature_batch is None or feature_batch.empty:
            unavailable_features += len(candidate_batch)
            continue
        feature_ids = set(feature_batch[ID_COLUMN].astype("string"))
        matched_candidates = matched_candidates.loc[
            matched_candidates[ID_COLUMN].astype("string").isin(feature_ids)
        ]
        unavailable_features += len(candidate_batch) - len(matched_candidates)

    if requires_markdown and not matched_candidates.empty:
        markdown_batch = load_spec_markdown(
            matched_candidates, data_dir=DATA_DIR, local_data_only=LOCAL_DATA_ONLY
        )
        markdown_ids = set(markdown_batch[ID_COLUMN].astype("string"))
        available_candidates = matched_candidates.loc[
            matched_candidates[ID_COLUMN].astype("string").isin(markdown_ids)
        ]
        unavailable_markdown += len(matched_candidates) - len(available_candidates)
        matched_candidates = available_candidates

    matched_candidates = matched_candidates.iloc[:needed]
    if matched_candidates.empty:
        continue
    matched_ids = set(matched_candidates[ID_COLUMN].astype("string"))
    selected_metadata_parts.append(matched_candidates)
    if requires_numeric:
        selected_feature_parts.append(
            feature_batch.loc[
                feature_batch[ID_COLUMN].astype("string").isin(matched_ids)
            ]
        )
    if requires_markdown:
        selected_markdown_parts.append(
            markdown_batch.loc[
                markdown_batch[ID_COLUMN].astype("string").isin(matched_ids)
            ]
        )
    selected_count += len(matched_candidates)

if selected_count < target_count:
    raise ValueError(
        f"Only {selected_count} of {target_count} requested specs have the "
        "required model inputs; no incomplete prediction file will be written."
    )
prediction_metadata = pd.concat(selected_metadata_parts, ignore_index=True)
if unavailable_features:
    print(f"Skipped {unavailable_features} specs without precomputed features.")
if unavailable_markdown:
    print(f"Skipped {unavailable_markdown} specs without readable Markdown.")
print(f"Random unlabeled specs selected: {len(prediction_metadata)}")

prediction_rows = prediction_metadata[[ID_COLUMN, REPOSITORY_COLUMN]].copy()
if requires_numeric:
    unlabeled_features = build_features(
        pd.concat(selected_feature_parts, ignore_index=True)
    )
    prediction_rows = prediction_rows.merge(
        unlabeled_features, on=ID_COLUMN, how="inner", validate="one_to_one"
    )
if requires_markdown:
    prediction_rows = prediction_rows.merge(
        pd.concat(selected_markdown_parts, ignore_index=True),
        on=ID_COLUMN,
        how="inner",
        validate="one_to_one",
    )
if len(prediction_rows) != target_count:
    raise RuntimeError("Prediction rows do not match the requested sample size.")

training_inputs = select_model_inputs(training_features, feature_columns)
prediction_inputs = select_model_inputs(prediction_rows, feature_columns)
final_model = clone(holdout_result["model"])
fit_baseline_model(
    final_model,
    training_inputs,
    training_labels,
    sample_weight=training_confidences.to_numpy() / 100,
)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
final_model_path = MODEL_DIR / f"{MODEL_NAME}_all_labeled.joblib"
joblib.dump(final_model, final_model_path)
print(f"Saved all-labeled prediction model to: {final_model_path}")
probability_ai, predicted_labels = predict_model(final_model, prediction_inputs)

predictions = pd.DataFrame(
    {
        SPEC_ID_COLUMN: prediction_rows[ID_COLUMN].astype("string"),
        REPOSITORY_COLUMN: prediction_rows[REPOSITORY_COLUMN],
        "predicted_label": predicted_labels,
        "probability_ai_assisted": probability_ai,
    }
)
predictions = predictions.apply(lambda column: column.map(escape_spreadsheet_formula))
PREDICTIONS_CSV.parent.mkdir(parents=True, exist_ok=True)
run_id = uuid.uuid4().hex
run_output = PREDICTIONS_CSV.with_name(
    f"{PREDICTIONS_CSV.stem}_{run_id}{PREDICTIONS_CSV.suffix}"
)
temporary_output = None
try:
    with tempfile.NamedTemporaryFile(
        mode="w",
        encoding="utf-8",
        newline="",
        dir=PREDICTIONS_CSV.parent,
        prefix=f".{PREDICTIONS_CSV.stem}_{run_id}_",
        suffix=".tmp",
        delete=False,
    ) as temporary_file:
        temporary_output = Path(temporary_file.name)
    predictions.to_csv(temporary_output, index=False)
    os.replace(temporary_output, run_output)
    temporary_output = None
finally:
    if temporary_output is not None:
        temporary_output.unlink(missing_ok=True)
print(f"Saved {len(predictions)} predictions to {run_output.name}")
display(predictions.head(20))